<a href="https://colab.research.google.com/github/jeffheaton/heaton-life/blob/main/python/examples/heaton_life_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# heaton-life

Emergence algorithms — cellular automata, Lenia, fractals, boids, and reaction-diffusion — as a NumPy library with one rendering pipeline. Every system is defined by a language-neutral specification and pinned by conformance vectors, so `(params, seed)` fully determines a run and the Python and .NET implementations agree bit for bit on the discrete automata.

You can install heaton-life with pip.

In [ ]:
!pip install --upgrade heaton-life

In [ ]:
import heaton_life as hl
from heaton_life.version import BUILD, BUILD_DATE, VERSION

print(f"heaton-life version: {hl.__version__}")
# BUILD is 0 for a local build; the Build Library workflow stamps the run number and date.
print(f"build stamp: {VERSION} build {BUILD} {BUILD_DATE}")

You may have to restart your Colab environment after the install.

## Life-like cellular automata

Any `B/S` rule string, a random "soup" start, and a seed. `animate` runs the simulation and captures frames; in a notebook the result displays inline as a GIF (use `.save("life.gif")` to write it to disk). Re-run the cell with the same seed and you get exactly the same animation — that is the determinism contract.

In [ ]:
sim = hl.ca.LifeLike("B3/S23", size=(256, 256), init="soup", seed=42)
hl.render.animate(sim, steps=160, every=2, cmap="phosphor", scale=2)

## MergeLife

MergeLife (Heaton, 2017) is a continuous-color cellular automaton whose entire rule is 32 hex digits — eight (range, percent) octet pairs. The library ships the featured-rule gallery that every implementation shares; the first entry is the rule from the paper.

In [ ]:
for featured in hl.ca.MERGELIFE_GALLERY:
    print(f"{featured.rule}  {featured.name}: {featured.description}")

In [ ]:
red_world = hl.ca.MERGELIFE_GALLERY[0]
sim = hl.ca.MergeLife(red_world.rule, size=(128, 128), seed=7)
# MergeLife frames are already RGB, so no colormap is involved.
hl.render.animate(sim, steps=300, every=3, scale=3)

Any 32-hex-digit rule string works, not just gallery entries — the gallery is only a curated way in. This one is "Beetle Meadow" from the gallery, passed as a plain rule string; try editing a few digits and re-running to see how different the physics becomes.

In [ ]:
sim = hl.ca.MergeLife("ea44-55df-9025-bead-5f6e-45ca-6168-275a", size=(128, 128), seed=7)
hl.render.animate(sim, steps=300, every=3, scale=3)

## Gray-Scott reaction-diffusion

Two chemicals on a torus; the `(feed, kill)` pair picks a point in the phase diagram. The presets are well-known coordinates. Patterns take a few thousand steps to form, so the simulation is advanced first and then a still frame and a short animation are rendered.

In [ ]:
print(list(hl.rd.GRAY_SCOTT_PRESETS))

preset = hl.rd.GRAY_SCOTT_PRESETS["Coral"]
sim = hl.rd.GrayScott(size=(256, 256), seed=3, **preset)
sim.step(2000)
hl.render.to_image(sim.frame(), cmap="ice", scale=2)

In [ ]:
hl.render.animate(sim, steps=1500, every=30, cmap="ice", scale=2)

## Fractals with deep zoom

Escape-time fractals render through a `Viewport` (center as decimal strings, zoom as a power of ten). Plain float64 pixelates near a zoom of 10¹²–10¹³; beyond that the library switches to perturbation with rebasing automatically, so this view at 10¹⁴ still resolves. The reference orbit is computed in arbitrary precision — install `heaton-life[precision]` for the faster gmpy2 path; the built-in mpmath fallback is what runs here.

One thing to know about deep zooms: the iteration cap is a horizon. Pixels that have not escaped by `max_iter` count as interior and render black, and the deeper the view, the more iterations its neighbourhood needs — a cap that is generous for the full set can black out an entire deep frame. If a deep render comes back solid black, raise `max_iter`.

In [ ]:
# At this depth every pixel takes thousands of iterations to escape: 3000 blacks out
# the whole frame, 5000 resolves it (the repository's gallery tile uses the same view).
frac = hl.fractal.Mandelbrot(max_iter=5000)
view = hl.Viewport(
    center_re="-0.743643887037158704752191506114774",
    center_im="0.131825904205311970493132056385139",
    zoom_log10=14.0,
)
field = frac.render((960, 540), view)
hl.render.to_image(field, cmap="fire")

A zoom movie dives from the full set toward a viewport; the same call with a `.mp4` target works after `pip install "heaton-life[video]"`.

In [ ]:
target = hl.Viewport(center_re="-0.7435", center_im="0.1314", zoom_log10=4.0)
hl.fractal.zoom_animation(frac, (320, 320), target, steps=60, cmap="fire")

## Where to go next

- The other families follow the same `step()` / `frame()` protocol: `hl.ca.Elementary`, `hl.ca.Cyclic`, `hl.ca.Wireworld`, `hl.lenia` (classic, asymptotic, flow), and `hl.boids`. `hl.render.list_colormaps()` lists the built-in colormaps.
- MergeLife rules can be **evolved** against the paper's objective, reproducibly from a seed — it is slow enough to deserve its own session:

  ```python
  from heaton_life.evolve import Evolver
  best = Evolver(size=(64, 64), population_size=20, seed=42).run(max_evals=200)
  print(best.genome, best.score)
  ```

- The interactive playground: `pip install "heaton-life[playground]"` then `heaton-life`.
- The same library for .NET: `dotnet add package HeatonLife.Core` ([NuGet](https://www.nuget.org/packages/HeatonLife.Core/)), a dependency-free `netstandard2.1` assembly held to the same conformance vectors, so a `(params, seed)` from this notebook gives the same run in C#.
- Specifications, conformance vectors, and both implementations: [github.com/jeffheaton/heaton-life](https://github.com/jeffheaton/heaton-life).